In [2]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib

In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


### Awesome oscillator for ticker - "AAPL"

In [2]:
df = yf.download(tickers="AAPL", start="2015-05-05", end="2025-05-05")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)


median_price = (df["High"] + df["Low"]) / 2

awesome = talib.SMA(median_price,timeperiod=5) - talib.SMA(median_price, timeperiod=34)


entries = (awesome < 0) & (awesome.shift(1) > 0)
exits = (awesome > 0) & (awesome.shift(1) < 0)



backtest = vbt.Portfolio.from_signals(
            median_price, 
            entries = entries,
            exits = exits,
            init_cash = 100000.0,
            size = 10000.0,
            size_type = 'value',
            freq = '1D',
            fees = 0.001)


backtest.plot().show()
print(backtest.stats())


[*********************100%***********************]  1 of 1 completed


Start                         2015-05-05 00:00:00
End                           2025-05-02 00:00:00
Period                         2515 days 00:00:00
Start Value                              100000.0
End Value                           100790.307303
Total Return [%]                         0.790307
Benchmark Return [%]                   620.631393
Max Gross Exposure [%]                  10.842684
Total Fees Paid                        881.671979
Max Drawdown [%]                         4.236267
Max Drawdown Duration          1640 days 00:00:00
Total Trades                                   44
Total Closed Trades                            44
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            72.727273
Best Trade [%]                           9.859538
Worst Trade [%]                        -24.329733
Avg Winning Trade [%]                    3.403844
Avg Losing Trade [%]                    -8.418328


### Loading and cleaning S and P 500 companies

In [4]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2025-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")


valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns)
)
close_df, high_df, low_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
)




[*                      3%                       ]  15 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**                     5%                       ]  25 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[************          24%                       ]  121 of 500 completed$Q: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[**********************86%****************       ]  429 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2025-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1746417600")
[*********************100%***********************]  500 of 500 co

### Awesome Oscillator for S and P 500 companies

In [13]:
median_price = (high_df + low_df) / 2


awesome_dict = {}

for ticker in median_price.columns:
    prices = median_price[ticker].to_numpy(dtype=np.float64)

    sma5 = talib.SMA(prices, timeperiod=5)
    sma34 = talib.SMA(prices, timeperiod=34)

    awesome_dict[ticker] = sma5 - sma34

awesome = pd.DataFrame(
    awesome_dict,
    index=median_price.index
)


entries = (awesome < 0) & (awesome.shift(1) > 0)
exits = (awesome > 0) & (awesome.shift(1) < 0)

backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=10_000.0,
    size_type="value",
    cash_sharing = True,
    group_by= True,
    freq="1D",
    fees=0.001
)

# 8. View results
print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               386669.831692
Total Return [%]                           286.669832
Benchmark Return [%]                       337.427317
Max Gross Exposure [%]                          100.0
Total Fees Paid                          55648.511352
Max Drawdown [%]                            40.381348
Max Drawdown Duration               414 days 00:00:00
Total Trades                                     4181
Total Closed Trades                              4138
Total Open Trades                                  43
Open Trade PnL                          -29350.810786
Win Rate [%]                                71.652972
Best Trade [%]                              53.949921
Worst Trade [%]                             -78.87564
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data



### Awesome Oscillator for 500 companies with stop loss 20 percent

In [19]:
backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=10_000.0,
    size_type="value",
    cash_sharing = True,
    group_by= True,
    freq="1D",
    fees=0.001,
    sl_stop=0.2,
)

# 8. View results
print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2025-05-02 00:00:00
Period                             2515 days 00:00:00
Start Value                                  100000.0
End Value                               372665.916159
Total Return [%]                           272.665916
Benchmark Return [%]                       337.427317
Max Gross Exposure [%]                          100.0
Total Fees Paid                          59181.846858
Max Drawdown [%]                            34.103994
Max Drawdown Duration               417 days 00:00:00
Total Trades                                     4484
Total Closed Trades                              4440
Total Open Trades                                  44
Open Trade PnL                            -704.199936
Win Rate [%]                                71.171171
Best Trade [%]                              53.949921
Worst Trade [%]                            -58.760255
Avg Winning Trade [%]       

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'orders' does not support grouped data

c:\Users\Haresh L\anaconda3\Lib\site-packages\vectorbt\generic\plots_builder.py:329: UserWarning:

Subplot 'trade_pnl' does not support grouped data

